# Praxis-SFT v2 — Treino QLoRA do Granite-4.1-8B em NVIDIA L4

PLAN.md, seções 11 e 12. Todas as configurações importantes vêm de `configs/train_l4.yaml`
— nenhuma célula abaixo deve conter um hiperparâmetro que não esteja também nesse arquivo.

Cada célula é escrita para ser re-executável isoladamente após a célula 4 (configuração de
diretórios), assumindo que os diretórios já existem — permite retomar depois de uma queda de
sessão do Colab sem re-rodar tudo do zero (seção 12.2).

**Status desta geração**: células 1-6 e 17 são executáveis como estão neste repositório.
Células 7-16 exigem GPU + as dependências de `requirements-train.txt` (Transformers, PEFT,
bitsandbytes, TRL) e por isso só rodam de fato dentro do Colab — nesta etapa de
desenvolvimento local elas documentam a implementação pretendida, mas não foram executadas.

## 1. Verificação da GPU

In [ ]:
import subprocess
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "configs" / "train_l4.yaml").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from src.training import TrainConfig

config = TrainConfig.load()

try:
    smi_output = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                                 capture_output=True, text=True, timeout=10)
    gpu_name = smi_output.stdout.strip()
except FileNotFoundError:
    gpu_name = ""

print(f"GPU detectada: {gpu_name!r}")
if config.require_gpu_name_contains not in gpu_name:
    raise RuntimeError(
        f"Esta configuração exige uma GPU cujo nome contenha "
        f"{config.require_gpu_name_contains!r} (train_l4.yaml); encontrado: {gpu_name!r}. "
        "Abortando para evitar rodar hiperparâmetros calibrados para L4 numa GPU diferente."
    )

## 2. Instalação de dependências (versões fixadas)

In [ ]:
# Seção 12.2 — sem !pip install sem pinning. Versões exatas devem ser fixadas aqui antes do
# treino real; requirements-train.txt lista os pacotes, não as versões travadas.
%pip install -q -r ../requirements-train.txt -r ../requirements.txt

## 3. Montagem opcional do Google Drive

In [ ]:
USE_DRIVE = config.run_on_colab

if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount(config.drive_mount_point)
    except ImportError:
        print("Não está rodando no Colab — pulando montagem do Drive (modo teste local).")
        USE_DRIVE = False

## 4. Configuração de diretórios (a partir do config, sem paths mágicos)

In [ ]:
output_dir = REPO_ROOT / config.output_dir
logs_dir = REPO_ROOT / config.logs_dir
adapter_dir = REPO_ROOT / config.adapter_dir

for d in (output_dir, logs_dir, adapter_dir):
    d.mkdir(parents=True, exist_ok=True)

print("output_dir:", output_dir)
print("logs_dir:", logs_dir)
print("adapter_dir:", adapter_dir)

## 5. Carregamento do dataset (train/validation já validados pelo pipeline offline — seção 9)

In [ ]:
import json

def load_split(path: Path) -> list[dict]:
    examples = []
    for file in sorted(path.glob("*.json")):
        with file.open("r", encoding="utf-8") as f:
            examples.append(json.load(f))
    return examples

train_examples = load_split(REPO_ROOT / config.train_path)
validation_examples = load_split(REPO_ROOT / config.validation_path)

print(f"train: {len(train_examples)} exemplos | validation: {len(validation_examples)} exemplos")
assert train_examples, (
    "data/train está vazio — rode scripts/generate_dataset.py e scripts/validate_dataset.py "
    "antes do treino (seção 9)"
)

## 6. Validação rápida de amostra (sanity check, não revalida o dataset)

In [ ]:
from collections import Counter

task_type_counts = Counter(ex["metadata"]["task_type"] for ex in train_examples)
print("Distribuição de task_type no split de treino:")
for task_type, count in task_type_counts.most_common():
    print(f"  {task_type}: {count}")

print("\nAmostra (primeiro exemplo):")
print(train_examples[0]["trajectory"]["raw_text"][:500])

## 7. Carregamento do Granite-4.1-8B em 4-bit

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=config.load_in_4bit,
    bnb_4bit_quant_type=config.bnb_4bit_quant_type,
    bnb_4bit_compute_dtype=getattr(torch, config.bnb_4bit_compute_dtype),
    bnb_4bit_use_double_quant=config.bnb_4bit_use_double_quant,
)

tokenizer = AutoTokenizer.from_pretrained(config.base_model)
model = AutoModelForCausalLM.from_pretrained(
    config.base_model, quantization_config=bnb_config, device_map="auto"
)

print("Módulos de atenção/MLP disponíveis (conferir contra lora_target_modules antes da célula 8):")
sample_module_names = {name.split(".")[-1] for name, _ in model.named_modules()}
print(sorted(sample_module_names & set(config.lora_target_modules)) or "NENHUM MATCH — revisar target_modules (risco seção 17)")

## 8. Configuração QLoRA

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)
if config.gradient_checkpointing:
    model.gradient_checkpointing_enable()

lora_config = LoraConfig(
    r=config.lora_r,
    lora_alpha=config.lora_alpha,
    lora_dropout=config.lora_dropout,
    target_modules=list(config.lora_target_modules),
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 9. Estimativa e monitoramento de VRAM (seção 11.3 — medir, não assumir)

In [ ]:
torch.cuda.reset_peak_memory_stats()
allocated_gb = torch.cuda.memory_allocated() / 1e9
print(f"VRAM alocada após carregar modelo + LoRA: {allocated_gb:.2f} GB")
print("Baseline experimental esperado (seção 11.3): ~12-17 GB de 24 GB — ajustar sequence_length/"
      "batch_size em train_l4.yaml se este número já estiver alto antes do treino começar.")

## 10. Loop de treinamento (SFTTrainer + máscara de loss por segmento, D7)

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

from src.training.data_collator import TrajectoryDataCollator

collator = TrajectoryDataCollator(tokenizer, max_length=config.sequence_length)

training_args = TrainingArguments(
    output_dir=str(output_dir),
    per_device_train_batch_size=config.batch_size_per_device,
    gradient_accumulation_steps=config.gradient_accumulation_steps,
    learning_rate=config.learning_rate,
    lr_scheduler_type=config.lr_scheduler_type,
    warmup_ratio=config.warmup_ratio,
    num_train_epochs=config.num_train_epochs,
    max_steps=config.max_steps,
    optim=config.optim,
    gradient_checkpointing=config.gradient_checkpointing,
    group_by_length=config.group_by_length,
    bf16=(config.mixed_precision == "bf16"),
    seed=config.seed,
    eval_strategy="steps",
    eval_steps=config.eval_steps,
    save_steps=config.save_steps,
    save_total_limit=config.save_total_limit,
    logging_dir=str(logs_dir),
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=[{"raw_text": ex["trajectory"]["raw_text"]} for ex in train_examples],
    eval_dataset=[{"raw_text": ex["trajectory"]["raw_text"]} for ex in validation_examples],
    data_collator=collator,
)

trainer.train()

## 11. Logging estruturado

In [ ]:
log_history_path = logs_dir / "trainer_log_history.json"
with log_history_path.open("w", encoding="utf-8") as f:
    json.dump(trainer.state.log_history, f, ensure_ascii=False, indent=2)
print(f"Histórico de treino salvo em {log_history_path}")

## 12. Checkpoints periódicos

In [ ]:
# Checkpoints já são salvos automaticamente por save_steps/save_total_limit (célula 10).
print("Checkpoints disponíveis:", sorted(p.name for p in output_dir.glob("checkpoint-*")))

## 13. Retomada a partir de checkpoint (testável isoladamente)

In [ ]:
RESUME_FROM_CHECKPOINT = None  # ex.: str(output_dir / "checkpoint-150")

if RESUME_FROM_CHECKPOINT:
    trainer.train(resume_from_checkpoint=RESUME_FROM_CHECKPOINT)

## 14. Avaliação rápida (probes reais, não só perplexidade)

In [ ]:
from src.inference.transformers_runner import TransformersModelRunner
from src.evaluation import probes, summarize
from src.security import SandboxContext, SandboxPolicy
from src.tools import ToolExecutorRegistry

# Nota: TransformersModelRunner.generate ainda não implementa geração real (ver seção 5.5 do
# módulo) — este bloco documenta o ponto de integração pretendido para quando isso existir.
print("Avaliação rápida via probes fica pendente até TransformersModelRunner.generate estar implementado.")

## 15. Salvamento do adapter final (não merge — D1 fora de escopo desta geração)

In [ ]:
model.save_pretrained(str(adapter_dir))
tokenizer.save_pretrained(str(adapter_dir))
print(f"Adapter salvo em {adapter_dir}")

## 16. Teste de inferência (trajetória completa via harness, não só geração crua)

In [ ]:
# Pendente da mesma implementação da célula 14 (TransformersModelRunner.generate real).
print("Pendente: rodar run_agent_loop real contra o adapter treinado (M6).")

## 17. Exportação de resultados

In [ ]:
outputs_summary = {
    "base_model": config.base_model,
    "adapter_dir": str(adapter_dir),
    "train_examples": len(train_examples),
    "validation_examples": len(validation_examples),
    "config_used": config.__dict__,
}
summary_path = REPO_ROOT / "outputs" / "run_summary.json"
summary_path.parent.mkdir(parents=True, exist_ok=True)
with summary_path.open("w", encoding="utf-8") as f:
    json.dump(outputs_summary, f, ensure_ascii=False, indent=2, default=str)
print(f"Resumo da execução salvo em {summary_path}")